In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/05 21:39:21 WARN Utils: Your hostname, Ethans-Laptop.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.4 instead (on interface en0)
26/03/05 21:39:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 21:39:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/05 21:39:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/05 21:39:21 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-05 21:39:46--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:234c:4a00:b:20a5:b140:21, 2600:9000:234c:8800:b:20a5:b140:21, 2600:9000:234c:400:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:234c:4a00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  21.0MB/s    in 3.2s    

2026-03-05 21:39:50 (21.0 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [6]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [7]:
df_yellow.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [8]:
df_yellow.repartition(4).write.parquet('data/pq/yellow/2025')

In [ ]:
# 25.6MB by checking the files in finder

In [9]:
from pyspark.sql import functions as F

In [16]:
##How many taxi trips were there on the 15th of November?

##Consider only trips that started on the 15th of November. 162,604
df_yellow.createOrReplaceTempView('yellow')

In [18]:
df_result = spark.sql (
    """
    select count(*) from yellow
    where date(tpep_pickup_datetime) ='2025-11-15'
    """
)

In [19]:
df_result.show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [24]:
## What is the length of the longest trip in the dataset in hours? 90.6
spark.sql (
    """
    select round(max(date_diff(hour,tpep_pickup_datetime,tpep_dropoff_datetime)),2)
    from yellow
    """
).show()

+-------------------------------------------------------------------------------+
|round(max(timestampdiff(hour, tpep_pickup_datetime, tpep_dropoff_datetime)), 2)|
+-------------------------------------------------------------------------------+
|                                                                             90|
+-------------------------------------------------------------------------------+



In [ ]:
##Spark's User Interface which shows the application's dashboard runs on which local port?

##use localhost:4040. 

In [25]:
df_zone = spark.read.parquet('zones')

In [26]:
df_zone.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [28]:
df_join =df_yellow.join(df_zone, df_yellow.PULocationID==df_zone.LocationID)

In [29]:
df_join.createOrReplaceTempView('df_join')

In [31]:
##Governor's Island...|       1|Eltingville/Annad...|       1||       Arden Heights|       1|

spark.sql(
    """
    select zone,count(*)
    from df_join
    group by zone
    order by count(*)
    """
).show()

+--------------------+--------+
|                zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|Eltingville/Annad...|       1|
|       Arden Heights|       1|
|       Port Richmond|       3|
|       Rikers Island|       4|
|   Rossville/Woodrow|       4|
| Green-Wood Cemetery|       4|
|         Great Kills|       4|
|         Jamaica Bay|       5|
|         Westerleigh|      12|
|        Crotona Park|      14|
|             Oakwood|      14|
|New Dorp/Midland ...|      14|
|       West Brighton|      14|
|       Willets Point|      15|
|Breezy Point/Fort...|      16|
|Saint George/New ...|      17|
|       Broad Channel|      18|
|     Mariners Harbor|      21|
|Heartland Village...|      22|
+--------------------+--------+
only showing top 20 rows
